# 03. Dashboard tipo RQ aplicado a contrataciones publicas

Este notebook arma una vista tipo dashboard con filtros simples por anio, departamento y categoria.

## Librerias

In [ ]:
# Librerias para el dashboard en notebook.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## Carga de datos y filtros

In [ ]:
# Leemos la base maestra.
BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'data'
df = pd.read_csv(DATA_DIR / 'contrataciones_peru_2022_2024_maestro.csv', low_memory=False)
df['n_postores'] = pd.to_numeric(df['n_postores'], errors='coerce')
df['monto_adjudicado'] = pd.to_numeric(df['monto_adjudicado'], errors='coerce').fillna(0)
df['monto_MM'] = df['monto_adjudicado'] / 1_000_000
df['un_solo_postor'] = df['un_solo_postor'].astype(str).str.lower().eq('true')
df['fecha_proceso'] = pd.to_datetime(df['fecha_proceso'], errors='coerce')
df['mes'] = df['fecha_proceso'].dt.to_period('M').astype(str)

# Estos filtros se pueden cambiar manualmente antes de correr el dashboard.
filtro_anios = [2022, 2023, 2024]
filtro_departamentos = []
filtro_categorias = []

dash = df.copy()
if filtro_anios:
    dash = dash[dash['anio'].isin(filtro_anios)]
if filtro_departamentos:
    dash = dash[dash['departamento'].isin(filtro_departamentos)]
if filtro_categorias:
    dash = dash[dash['categoria'].isin(filtro_categorias)]

print('Procesos filtrados:', dash['ocid'].nunique())


## KPIs principales

In [ ]:
# Mostramos los indicadores principales del tablero.
print('Total procesos:', f"{dash['ocid'].nunique():,}")
print('% un solo postor:', f"{dash['un_solo_postor'].mean() * 100:.2f}%")
print('Monto adjudicado total (MM PEN):', f"{dash['monto_MM'].sum():,.2f}")
print('% contrataciones directas:', f"{((dash['metodo_simple'] == 'Directa').mean() * 100):.2f}%")


## Vista 1. Distribucion por categoria

In [ ]:
# Treemap del monto por categoria para una vista general del tablero.
g1 = dash.groupby('categoria', as_index=False).agg(monto_MM=('monto_MM', 'sum'))
fig = px.treemap(g1, path=['categoria'], values='monto_MM', title='Monto adjudicado por categoria')
fig.show()


## Vista 2. Riesgo por departamento

In [ ]:
# Barras con los departamentos de mayor concentracion de procesos con un solo postor.
g2 = (dash[dash['departamento'] != 'Sin dato']
      .groupby('departamento', as_index=False)
      .agg(casos_un_postor=('un_solo_postor', 'sum'))
      .sort_values('casos_un_postor', ascending=False)
      .head(15))
plt.figure(figsize=(10, 6))
plt.barh(g2['departamento'], g2['casos_un_postor'], color='darkorange')
plt.title('Departamentos con mayor riesgo por un solo postor')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Vista 3. Competencia por categoria

In [ ]:
# Boxplot para comparar el numero de postores entre categorias.
g3 = dash[dash['n_postores'].notna()].copy()
g3['n_postores'] = g3['n_postores'].clip(upper=15)
plt.figure(figsize=(10, 5))
sns.boxplot(data=g3, x='categoria', y='n_postores', showfliers=False)
plt.title('Competencia por categoria')
plt.tight_layout()
plt.show()


## Vista 4. Tendencia mensual por metodo

In [ ]:
# Serie temporal para seguir la evolucion de procesos competitivos y directos.
g4 = (dash[dash['metodo_simple'].isin(['Competitivo', 'Directa'])]
      .groupby(['mes', 'metodo_simple'], as_index=False)
      .agg(procesos=('ocid', 'nunique')))
g4['mes_dt'] = pd.to_datetime(g4['mes'] + '-01')
g4 = g4.sort_values('mes_dt')
plt.figure(figsize=(12, 5))
sns.lineplot(data=g4, x='mes_dt', y='procesos', hue='metodo_simple', marker='o')
plt.title('Evolucion mensual por metodo de contratacion')
plt.tight_layout()
plt.show()
